# Index Restoration Utility

Restore the pre-configured Azure AI Search indexes (`hrdocs` and `healthdocs`) used throughout LAB532. The refreshed notebooks assume these indexes already exist in the `magottei-eastus2euap` search service.

In [ ]:
import json
import os
import traceback

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import AzureCliCredential
from azure.search.documents.aio import SearchClient
from azure.search.documents.indexes.aio import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex

TENANT_ID = "72f988bf-86f1-41af-91ab-2d7cd011db47"
load_dotenv(override=True)
SEARCH_ENDPOINT = os.environ.get("AZURE_SEARCH_SERVICE_ENDPOINT") or os.environ.get("MAGOTTEI_EASTUS2EUAP_ENDPOINT")
SEARCH_KEY = os.environ.get("AZURE_SEARCH_ADMIN_KEY") or os.environ.get("MAGOTTEI_EASTUS2EUAP_KEY")
SEARCH_AUTH_MODE = os.environ.get("AZURE_SEARCH_AUTH_MODE", "aad").lower()
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]

if not SEARCH_ENDPOINT:
    raise RuntimeError("Set AZURE_SEARCH_SERVICE_ENDPOINT in notebooks/.env")
if SEARCH_AUTH_MODE == "api-key" and not SEARCH_KEY:
    raise RuntimeError("AZURE_SEARCH_AUTH_MODE=api-key requires AZURE_SEARCH_ADMIN_KEY in notebooks/.env")

credential = AzureKeyCredential(SEARCH_KEY) if SEARCH_AUTH_MODE == "api-key" else AzureCliCredential(tenant_id=TENANT_ID)
print(f"Search endpoint : {SEARCH_ENDPOINT}")
print(f"Search auth     : {SEARCH_AUTH_MODE}")
print(f"OpenAI endpoint : {AZURE_OPENAI_ENDPOINT}")

In [ ]:
async def restore_index(endpoint: str, credential, index_name: str, index_file: str, records_file: str, azure_openai_endpoint: str):
    default_path = r"../data/index-data"
    try:
        print(f"[{index_name}] Starting index restoration...")
        async with SearchIndexClient(endpoint=endpoint, credential=credential) as client:
            index_file_path = os.path.join(default_path, index_file)
            print(f"[{index_name}] Reading index definition from: {index_file_path}")
            with open(index_file_path, "r", encoding="utf-8") as in_file:
                index = SearchIndex.deserialize(json.load(in_file))
                index.name = index_name
                index.vector_search.vectorizers[0].parameters.resource_url = azure_openai_endpoint
                index.vector_search.vectorizers[0].parameters.api_key = AZURE_OPENAI_KEY
                await client.create_or_update_index(index)
        async with SearchClient(endpoint=endpoint, index_name=index_name, credential=credential) as client:
            records_file_path = os.path.join(default_path, records_file)
            print(f"[{index_name}] Reading documents from: {records_file_path}")
            records, total_uploaded, batch_count = [], 0, 0
            with open(records_file_path, "r", encoding="utf-8") as in_file:
                for line_num, line in enumerate(in_file, 1):
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError as e:
                        print(f"[{index_name}] WARNING: Skipping invalid JSON on line {line_num}: {e}")
                        continue
                    if len(records) >= 100:
                        batch_count += 1
                        print(f"[{index_name}] Uploading batch #{batch_count} ({len(records)} documents)...")
                        await client.upload_documents(documents=records)
                        total_uploaded += len(records)
                        records = []
            if records:
                batch_count += 1
                print(f"[{index_name}] Uploading final batch #{batch_count} ({len(records)} documents)...")
                await client.upload_documents(documents=records)
                total_uploaded += len(records)
        print(f"[{index_name}] SUCCESS - Index restored. Total documents uploaded: {total_uploaded}")
        return True
    except Exception as e:
        print(f"[{index_name}] ERROR - {type(e).__name__}: {e}")
        traceback.print_exc()
        return False

In [ ]:
print("\n--- Processing hrdocs index ---")
await restore_index(SEARCH_ENDPOINT, credential, "hrdocs", "index.json", "hrdocs-exported.jsonl", AZURE_OPENAI_ENDPOINT)

In [ ]:
print("\n--- Processing healthdocs index ---")
await restore_index(SEARCH_ENDPOINT, credential, "healthdocs", "index.json", "healthdocs-exported.jsonl", AZURE_OPENAI_ENDPOINT)

## Next steps

Once both indexes are restored successfully, start with [Part 1: Multi-source KB with restored search indexes](part1-standard-foundry-iq-kb.ipynb).